# ДЗ №3. Оценка baseline и основной модели

Сравниваем модели по одному протоколу:

- **YOLOv8n** — предобученный baseline;
- **YOLO11n** — предобученный, показывает вклад архитектуры;
- **YOLO11n-home** — дообученный на 10 домашних классах COCO (`02_train_yolo11n.ipynb`);
- **YOLO11n-hardneg** — дообученный с трудными негативами Open Images train
  (`04_train_yolo11n_hardneg.ipynb`), если его вывод подключён к ноутбуку.

Каждая модель проверяется на размерах входа 640, 416 и 320 и на двух наборах:
COCO val2017 (протокол ДЗ №2, пороги по фолдам) и Open Images V7 test
(перенос на другой источник; пороги взяты с COCO и здесь не подбираются).

**Настройки Kaggle:** GPU T4, Internet — On.
**Входные данные:** COCO 2017 val (как в ДЗ №2) и выводы ноутбуков обучения.

In [ ]:
import subprocess
import sys

# Ветка или коммит репозитория с пакетом photoquest_eval.
REPO_REF = "hw_3"
_ = subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"photoquest-eval[yolo] @ git+https://github.com/plekhovrodion/mlsd@{REPO_REF}",
    ]
)
REPO_COMMIT = subprocess.check_output(
    ["git", "ls-remote", "https://github.com/plekhovrodion/mlsd", REPO_REF], text=True
).split()[0]
print("photoquest_eval:", REPO_REF, REPO_COMMIT)

In [ ]:
import hashlib
import importlib.metadata
import json
import platform
import shutil  # noqa: F401 — ячейка общая для обоих ноутбуков
import time
import urllib.request
import zipfile
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from photoquest_eval.config import (  # noqa: F401 — ячейка общая для обоих ноутбуков
    HOME_CLASSES,
    IMAGE_SIZES,
    SEED,
    TARGETS,
    YOLO_SETTINGS,
)

STARTED_UTC = datetime.now(UTC).isoformat()
STARTED = time.perf_counter()
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Kaggle Settings → Accelerator → GPU T4")
print("GPU:", torch.cuda.get_device_name(0), "| PyTorch:", torch.__version__)


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def package_versions(*names):
    return {name: importlib.metadata.version(name) for name in names}

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from PIL import Image

from photoquest_eval.data import (
    OPENIMAGES_URLS,
    copy_images,
    download_openimages,
    load_coco,
    load_openimages,
    sample_openimages,
)
from photoquest_eval.evaluate import coco_oof, fit_thresholds, transfer_metrics
from photoquest_eval.inference import predict_scores
from photoquest_eval.metrics import paired_bootstrap_diff

OUTPUT = Path("/kaggle/working/hw3_eval")
OUTPUT.mkdir(parents=True, exist_ok=True)
RULE_VERSION = "hw3-r1"
OI_SAMPLE = {"positives_per_class": 400, "negatives_per_class": 800, "seed": SEED}

## Данные

### COCO val2017

In [ ]:
import kagglehub

COCO_ROOT = Path(kagglehub.dataset_download("nikitamanaenkov/microsoft-coco-2017"))
VAL_ANNOTATIONS = sorted(COCO_ROOT.rglob("instances_val2017.json"))[0]
assert sha256(VAL_ANNOTATIONS) == (
    "e8c7f7908f1d7278341fae127d0da654f102f11bd7b21d8aeefa635b8c810b6f"
), "Снимок аннотаций отличается от ДЗ №2"
val_root = VAL_ANNOTATIONS.parent.parent
INPUT_VAL_IMAGES = next(
    p for p in (val_root / "val2017", val_root / "images" / "val2017") if p.is_dir()
)
# Оценка проходит по val много раз (модели × 3 размера), поэтому оригиналы один раз
# копируются на локальный диск без изменений, как в ДЗ №2.
VAL_IMAGES = Path("/tmp/coco_val2017")
failed_copies = copy_images(sorted(INPUT_VAL_IMAGES.glob("*.jpg")), VAL_IMAGES)
assert not failed_copies, failed_copies[:5]
coco = load_coco(VAL_ANNOTATIONS, VAL_IMAGES, TARGETS)
print(
    len(coco.image_ids),
    "фото;",
    "положительных по классам:",
    dict(zip(TARGETS, coco.labels.sum(axis=0).tolist(), strict=True)),
)

### Open Images V7 test

Выборка фиксируется до запуска моделей: для каждого класса до 400 фото с проверенной
положительной меткой и до 800 с проверенной отрицательной. Фото без проверенной
метки класса в его оценке не участвуют. При скачивании фото уменьшаются до 1024 px
по длинной стороне, чтобы уложиться в лимит диска Kaggle.

In [ ]:
OI_DIR = Path("/kaggle/working/openimages")
OI_DIR.mkdir(exist_ok=True)
csv_paths = {}
for key, name in (
    ("image_labels", "test-human-imagelabels.csv"),
    ("boxes", "test-annotations-bbox.csv"),
):
    csv_paths[key] = OI_DIR / name
    if not csv_paths[key].exists():
        urllib.request.urlretrieve(OPENIMAGES_URLS[key], csv_paths[key])
META_URL = "https://storage.googleapis.com/openimages/2018_04/test/test-images-with-rotation.csv"
csv_paths["meta"] = OI_DIR / "test-images-with-rotation.csv"
if not csv_paths["meta"].exists():
    urllib.request.urlretrieve(META_URL, csv_paths["meta"])
{key: sha256(path) for key, path in csv_paths.items()}

In [ ]:
oi_ids = sample_openimages(csv_paths["image_labels"], TARGETS, **OI_SAMPLE)
failed = download_openimages(oi_ids, OI_DIR / "images")
print(f"Выбрано {len(oi_ids):,} фото, не скачалось {len(failed)}")
oi = load_openimages(csv_paths["image_labels"], csv_paths["boxes"], OI_DIR / "images", TARGETS)
oi_meta = pd.read_csv(csv_paths["meta"]).set_index("ImageID")
manifest = oi_meta.reindex(oi.image_ids)[
    ["License", "Author", "OriginalLandingURL", "Rotation"]
].reset_index(names="ImageID")
manifest.to_csv(OUTPUT / "openimages_manifest.csv", index=False)
known = oi.labels >= 0
display(
    pd.DataFrame(
        {
            "positive_images": (oi.labels == 1).sum(axis=0),
            "verified_negatives": (oi.labels == 0).sum(axis=0),
            "unknown": (~known).sum(axis=0),
        },
        index=TARGETS,
    )
)
print("Фото с ненулевым Rotation:", int(manifest["Rotation"].fillna(0).ne(0).sum()))

## Модели и score

In [ ]:
CHECKPOINTS = {
    "yolov8n.pt": (
        "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8n.pt",
        "f59b3d833e2ff32e194b5bb8e08d211dc7c5bdf144b90d2c8412c47ccfc83b36",
    ),
    "yolo11n.pt": (
        "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt",
        "0ebbc80d4a7680d14987a577cd21342b65ecfd94632bd9a8da63ae6417644ee1",
    ),
}


def get_checkpoint(filename: str, folder: Path) -> Path:
    """Скачивает предобученные веса и сверяет SHA-256 с прогоном ДЗ №2."""
    url, expected = CHECKPOINTS[filename]
    path = folder / filename
    folder.mkdir(parents=True, exist_ok=True)
    if not path.is_file():
        urllib.request.urlretrieve(url, path)
    if sha256(path) != expected:
        raise ValueError(f"Не совпал SHA-256 весов {filename}")
    return path

In [ ]:
FINE_TUNED = {}
for name in ("yolo11n_home", "yolo11n_hardneg"):
    found = sorted(Path("/kaggle/input").rglob(f"{name}.pt"))
    if found:
        FINE_TUNED[name] = found[0]
if not FINE_TUNED:
    raise FileNotFoundError("Добавьте во входные данные вывод ноутбука обучения (02 или 04)")
# Разбор ошибок — для последней дообученной модели.
MAIN_MODEL = list(FINE_TUNED)[-1]
MODELS = {
    "yolov8n": get_checkpoint("yolov8n.pt", OUTPUT / "pretrained"),
    "yolo11n": get_checkpoint("yolo11n.pt", OUTPUT / "pretrained"),
    **FINE_TUNED,
}
model_hashes = {name: sha256(path) for name, path in MODELS.items()}
model_hashes

In [ ]:
scores = {}
timings = []
for name, weights in MODELS.items():
    for size in IMAGE_SIZES:
        for dataset in (coco, oi):
            started = time.perf_counter()
            scores[(dataset.name, name, size)] = predict_scores(
                weights, dataset.image_paths, TARGETS, size, device=0, **YOLO_SETTINGS
            )
            seconds = time.perf_counter() - started
            timings.append(
                {
                    "dataset": dataset.name,
                    "model": name,
                    "image_size": size,
                    "images": len(dataset.image_ids),
                    "seconds": seconds,
                }
            )
            print(f"{dataset.name:10} {name:13} {size}: {seconds:.0f} с")
np.savez_compressed(
    OUTPUT / "scores.npz",
    **{"__".join(map(str, key)): value for key, value in scores.items()},
)
pd.DataFrame(timings).to_csv(OUTPUT / "timings_t4.csv", index=False)

## COCO val2017: протокол ДЗ №2

Пороги подбираются по отрицательным примерам четырёх фолдов и проверяются на пятом.
Интервалы — перцентильный bootstrap по фото (1 000 повторов).

In [ ]:
coco_rows, calibration_rows = [], []
for name in MODELS:
    for size in IMAGE_SIZES:
        rows, calibration, oof = coco_oof(coco, scores[("coco", name, size)], name, size)
        coco_rows.extend(rows)
        calibration_rows.extend(calibration)
        np.savez_compressed(OUTPUT / f"coco_oof_{name}_{size}.npz", **oof)
coco_metrics = pd.DataFrame(coco_rows)
coco_metrics.to_csv(OUTPUT / "coco_per_class.csv", index=False)
pd.DataFrame(calibration_rows).to_csv(OUTPUT / "coco_calibration_folds.csv", index=False)


def macro(table):
    return (
        table.groupby(["dataset", "model", "image_size", "protocol"], sort=False)
        .agg(
            macro_ap=("average_precision", "mean"),
            macro_auc=("roc_auc", "mean"),
            recall=("recall", "mean"),
            far=("far", "mean"),
            classes_far_above_5pct=("far_exceeds_target", "sum"),
            recall_main=("recall_main", "mean"),
            recall_background=("recall_background", "mean"),
            recall_small_only=("recall_small_only", "mean"),
        )
        .reset_index()
    )


coco_summary = macro(coco_metrics)
display(coco_summary.round(4))

## Пороги для переноса и для игры

Итоговые пороги считаются по всем отрицательным фото COCO val2017 с консервативным
протоколом `confidence_95pct`. Для дообученных моделей они сохраняются
в `thresholds_<model>_<size>.json`: такой файл в ДЗ №4 загрузит браузер.

In [ ]:
thresholds = {}
for name in MODELS:
    for size in IMAGE_SIZES:
        thresholds[(name, size)] = fit_thresholds(
            coco,
            scores[("coco", name, size)],
            f"{name}-{model_hashes[name][:12]}",
            size,
            RULE_VERSION,
        )
for name in FINE_TUNED:
    for size in IMAGE_SIZES:
        thresholds[(name, size)].to_json(OUTPUT / f"thresholds_{name}_{size}.json")
print((OUTPUT / f"thresholds_{MAIN_MODEL}_640.json").read_text())

## Open Images V7 test: перенос

Негативы — только фото с проверенной отрицательной меткой. Срезы положительных фото:
`main` — рамка не меньше 5% кадра и в центральной зоне, `background` — остальные,
`small_only` — все рамки класса меньше 1% кадра.

In [ ]:
oi_rows, oi_predictions = [], {}
for name in MODELS:
    for size in IMAGE_SIZES:
        rows, predicted = transfer_metrics(
            oi, scores[("openimages", name, size)], thresholds[(name, size)], name
        )
        oi_rows.extend(rows)
        oi_predictions[(name, size)] = predicted
oi_metrics = pd.DataFrame(oi_rows)
oi_metrics.to_csv(OUTPUT / "openimages_per_class.csv", index=False)
oi_summary = macro(oi_metrics)
summary = pd.concat([coco_summary, oi_summary], ignore_index=True)
summary.to_csv(OUTPUT / "model_metrics.csv", index=False)
display(oi_summary.round(4))

## Значимость различий

Парный bootstrap на одних и тех же фото Open Images (размер входа 640):
разница recall и FAR моделей по каждому классу с 95% интервалом.

In [ ]:
pairs = [("yolo11n", "yolov8n")]
for name in FINE_TUNED:
    pairs += [(name, "yolov8n"), (name, "yolo11n")]
if len(FINE_TUNED) == 2:
    pairs.append(("yolo11n_hardneg", "yolo11n_home"))
diff_rows = []
for a, b in pairs:
    for column, target in enumerate(TARGETS):
        known = oi.mask(column)
        y = oi.labels[known, column] == 1
        diff = paired_bootstrap_diff(
            y, oi_predictions[(a, 640)][target][known], oi_predictions[(b, 640)][target][known]
        )
        for metric in ("recall", "far"):
            estimate, low, high = diff[metric]
            diff_rows.append(
                {
                    "model_a": a,
                    "model_b": b,
                    "target": target,
                    "metric": metric,
                    "diff": estimate,
                    "low": low,
                    "high": high,
                    "significant": not (low <= 0 <= high),
                }
            )
significance = pd.DataFrame(diff_rows)
significance.to_csv(OUTPUT / "openimages_paired_bootstrap.csv", index=False)
display(
    significance.groupby(["model_a", "model_b", "metric"])
    .agg(mean_diff=("diff", "mean"), significant_classes=("significant", "sum"))
    .round(4)
)

## Графики

In [ ]:
conservative = oi_metrics[oi_metrics["protocol"].eq("confidence_95pct")]
part = conservative[conservative["image_size"].eq(640)]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for axis, metric, title, upper, cmap in (
    (axes[0], "recall", "Recall на Open Images", 1.0, "Blues"),
    (axes[1], "far", "FAR на Open Images", 0.1, "OrRd"),
):
    table = part.pivot(index="target", columns="model", values=metric).reindex(TARGETS)
    image = axis.imshow(table, vmin=0, vmax=upper, aspect="auto", cmap=cmap)
    axis.set_xticks(range(len(table.columns)), table.columns, rotation=20)
    axis.set_yticks(range(len(TARGETS)), table.index)
    axis.set_title(title)
    for r in range(len(TARGETS)):
        for c in range(len(table.columns)):
            value = table.iloc[r, c]
            axis.text(
                c,
                r,
                f"{value:.1%}",
                ha="center",
                va="center",
                fontsize=8,
                color="white" if value > upper * 0.6 else "black",
            )
    fig.colorbar(image, ax=axis, shrink=0.8, format=PercentFormatter(1))
fig.suptitle("Перенос на Open Images, вход 640, консервативный порог")
fig.tight_layout()
fig.savefig(OUTPUT / "openimages_recall_far.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
cons = summary[summary["protocol"].eq("confidence_95pct")]
for axis, dataset in zip(axes, ("coco", "openimages"), strict=True):
    for name in MODELS:
        rows = cons[cons["dataset"].eq(dataset) & cons["model"].eq(name)]
        axis.plot(rows["image_size"], rows["recall"], marker="o", label=name)
    axis.set(title=f"Recall от размера входа — {dataset}", xlabel="imgsz", ylabel="recall")
    axis.yaxis.set_major_formatter(PercentFormatter(1))
    axis.legend()
fig.tight_layout()
fig.savefig(OUTPUT / "recall_by_image_size.png", dpi=150)
plt.show()

slices = oi_summary[oi_summary["image_size"].eq(640)].set_index("model")[
    ["recall_main", "recall_background", "recall_small_only"]
]
ax = slices.plot.bar(figsize=(8, 4), rot=0, title="Recall по срезам, Open Images, 640")
ax.yaxis.set_major_formatter(PercentFormatter(1))
plt.tight_layout()
plt.savefig(OUTPUT / "openimages_slices.png", dpi=150)
plt.show()

## Разбор ошибок дообученной модели

Для рюкзака, телефона и бутылки берём на Open Images (640) ложное подтверждение
с наибольшим score и пропуск с наименьшим. Авторы и лицензии показанных фото —
в `error_cases.csv`.

In [ ]:
ERROR_TARGETS = ["backpack", "cell phone", "bottle"]
oi_scores = scores[("openimages", MAIN_MODEL, 640)]
records = []
for target in ERROR_TARGETS:
    column = TARGETS.index(target)
    known = oi.mask(column)
    truth = oi.labels[:, column] == 1
    predicted = oi_predictions[(MAIN_MODEL, 640)][target]
    for kind, mask in (
        ("false_positive", predicted & ~truth & known),
        ("false_negative", ~predicted & truth & known),
    ):
        index = np.flatnonzero(mask)
        for row in index[np.argsort(oi_scores[index, column], kind="stable")]:
            records.append(
                {
                    "target": target,
                    "error_type": kind,
                    "image_id": oi.image_ids[row],
                    "score": float(oi_scores[row, column]),
                    "cutoff": thresholds[(MAIN_MODEL, 640)].rules[target].high,
                }
            )
error_cases = pd.DataFrame(records).merge(manifest, left_on="image_id", right_on="ImageID")
error_cases.to_csv(OUTPUT / "error_cases.csv", index=False)
examples = pd.concat(
    [
        error_cases[error_cases["error_type"].eq("false_positive")]
        .sort_values("score", ascending=False)
        .groupby("target")
        .head(1),
        error_cases[error_cases["error_type"].eq("false_negative")]
        .sort_values("score")
        .groupby("target")
        .head(1),
    ]
).sort_values(["target", "error_type"])
fig, axes = plt.subplots(len(ERROR_TARGETS), 2, figsize=(12, 11))
for axis, row in zip(axes.flat, examples.itertuples(), strict=True):
    with Image.open(OI_DIR / "images" / f"{row.image_id}.jpg") as image:
        axis.imshow(image.convert("RGB"))
    kind = "Ложное подтверждение" if row.error_type == "false_positive" else "Пропуск"
    axis.set_title(
        f"{MAIN_MODEL}: {row.target} — {kind}\n{row.image_id}, score {row.score:.3f}, "
        f"порог {row.cutoff:.3f}\n© {row.Author}, CC BY 2.0",
        fontsize=9,
    )
    axis.axis("off")
fig.tight_layout()
fig.savefig(OUTPUT / "error_examples.jpg", dpi=110, pil_kwargs={"quality": 85})
plt.show()
display(error_cases.groupby(["target", "error_type"]).size().unstack())

## Совпадение ONNX и PyTorch

Для ДЗ №4 модель будет работать в браузере из ONNX. Сравниваем score и решения
ONNX-версий дообученных моделей с PyTorch на 300 фото COCO val2017.

In [ ]:
rng = np.random.default_rng(SEED)
sample = sorted(rng.choice(len(coco.image_paths), 300, replace=False).tolist())
sample_paths = [coco.image_paths[i] for i in sample]
parity_rows = []
for name, weights in FINE_TUNED.items():
    for size in IMAGE_SIZES:
        onnx_path = weights.parent / f"{name}_{size}.onnx"
        onnx_scores = predict_scores(
            onnx_path, sample_paths, TARGETS, size, batch_size=1, device="cpu", **YOLO_SETTINGS
        )
        torch_scores = scores[("coco", name, size)][sample]
        rule = thresholds[(name, size)]
        cutoffs = np.array([rule.rules[t].high for t in TARGETS])
        parity_rows.append(
            {
                "model": name,
                "image_size": size,
                "max_abs_score_diff": float(np.abs(onnx_scores - torch_scores).max()),
                "decision_agreement": float(
                    ((onnx_scores > cutoffs) == (torch_scores > cutoffs)).mean()
                ),
                "onnx_sha256": sha256(onnx_path),
            }
        )
parity = pd.DataFrame(parity_rows)
parity.to_csv(OUTPUT / "onnx_parity.csv", index=False)
display(parity)

In [ ]:
metadata = {
    "repo_ref": REPO_REF,
    "repo_commit": REPO_COMMIT,
    "targets": list(TARGETS),
    "rule_version": RULE_VERSION,
    "yolo_settings": YOLO_SETTINGS,
    "image_sizes": list(IMAGE_SIZES),
    "openimages_sample": OI_SAMPLE,
    "openimages_images": len(oi.image_ids),
    "openimages_failed_downloads": failed,
    "openimages_csv_sha256": {key: sha256(path) for key, path in csv_paths.items()},
    "coco_val_images": len(coco.image_ids),
    "model_sha256": model_hashes,
    "gpu": torch.cuda.get_device_name(0),
    "python": platform.python_version(),
    "packages": package_versions(
        "torch", "ultralytics", "numpy", "pandas", "scikit-learn", "scipy", "onnxruntime"
    ),
    "started_utc": STARTED_UTC,
    "elapsed_seconds": round(time.perf_counter() - STARTED, 1),
}
(OUTPUT / "metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False))
with zipfile.ZipFile("/kaggle/working/hw3_eval_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUTPUT.rglob("*")):
        if path.is_file() and "pretrained" not in path.relative_to(OUTPUT).parts:
            z.write(path, path.relative_to(OUTPUT))
print(json.dumps(metadata, indent=2, ensure_ascii=False))